In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.pipeline_logs (
    log_timestamp TIMESTAMP,
    layer STRING,
    table_name STRING,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    record_count LONG,
    message STRING
)
USING DELTA

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS ecommerce.silver.export_logs;

In [0]:
export_log_path = "/Volumes/ecommerce/silver/export_logs/silver_pipeline.log"

In [0]:
from pyspark.sql.functions import current_timestamp
from datetime import datetime

def log_pipeline(layer, table_name, status, start_time, end_time, record_count, message):

    log_data = [(layer, table_name, status, start_time, end_time, record_count, message)]

    columns = [
        "layer",
        "table_name",
        "status",
        "start_time",
        "end_time",
        "record_count",
        "message"
    ]

    log_df = spark.createDataFrame(log_data, columns) \
        .withColumn("log_timestamp", current_timestamp())

    log_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.silver.pipeline_logs")

    export_log_path = f"/Volumes/ecommerce/silver/export_logs/{layer}_pipeline.log"

    log_string = f"{datetime.now()} | {layer} | {table_name} | {status} | {record_count} | {message}\n"

    try:
        existing_log = dbutils.fs.head(export_log_path)
        new_log = existing_log + log_string
    except:
        new_log = log_string

    dbutils.fs.put(export_log_path, new_log, True)from pyspark.sql.functions import current_timestamp
from datetime import datetime

def log_pipeline(layer, table_name, status, start_time, end_time, record_count, message):

    log_data = [(layer, table_name, status, start_time, end_time, record_count, message)]

    columns = [
        "layer",
        "table_name",
        "status",
        "start_time",
        "end_time",
        "record_count",
        "message"
    ]

    log_df = spark.createDataFrame(log_data, columns) \
        .withColumn("log_timestamp", current_timestamp())

    log_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.silver.pipeline_logs")

    export_log_path = f"/Volumes/ecommerce/silver/export_logs/{layer}_pipeline.log"

    log_string = f"{datetime.now()} | {layer} | {table_name} | {status} | {record_count} | {message}\n"

    try:
        existing_log = dbutils.fs.head(export_log_path)
        new_log = existing_log + log_string
    except:
        new_log = log_string

    dbutils.fs.put(export_log_path, new_log, True)

###==================================================
##CUSTOMER TABLE
###==================================================

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.customers (
    customer_id STRING NOT NULL, --Primary Key, Foreign Key
    customer_unique_id STRING NOT NULL, 
    customer_zip_code_prefix INT NOT NULL,
    customer_city STRING,
    customer_state STRING,
    create_date TIMESTAMP,
    update_date TIMESTAMP,
    file_location STRING
)
USING DELTA

In [0]:
%sql
ALTER TABLE ecommerce.silver.customers
DROP CONSTRAINT pk_customers CASCADE;

In [0]:
%sql
ALTER TABLE ecommerce.silver.customers
ADD CONSTRAINT pk_customers
PRIMARY KEY (customer_id);

In [0]:
from pyspark.sql.functions import col, trim, upper, current_timestamp
from datetime import datetime

start_time = datetime.now()

try:

    customers = spark.table("ecommerce.bronze.customers")

    silver_customers = (
        customers
        
        # Remove null primary key
        .filter(col("customer_id").isNotNull())
        
        # Remove duplicates
        .dropDuplicates(["customer_id"])

        .filter(col("customer_unique_id").isNotNull())
        .filter(col("customer_zip_code_prefix").isNotNull())
        
        # Standardization
        .withColumn("customer_city", upper(trim(col("customer_city"))))
        .withColumn("customer_state", upper(trim(col("customer_state"))))
        
        # Metadata columns
        .withColumn("create_date", current_timestamp())
        .withColumn("update_date", current_timestamp())
        .withColumn("file_location", col("source_file"))
        
        # Select final columns
        .select(
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state",
            "create_date",
            "update_date",
            "file_location"
        )
    )

    record_count = silver_customers.count()

    silver_customers.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce.silver.customers")

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "customers",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "Customers table loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "customers",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###==================================================
##ORDERS TABLE
###==================================================

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.orders (
    order_id STRING NOT NULL, -- Primary Key
    customer_id STRING NOT NULL, -- Foreign Key
    order_status STRING,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    create_date TIMESTAMP,
    update_date TIMESTAMP,
    file_location STRING
)
USING DELTA

In [0]:
%sql
ALTER TABLE ecommerce.silver.orders
DROP CONSTRAINT pk_orders CASCADE;

In [0]:
%sql
ALTER TABLE ecommerce.silver.orders
ADD CONSTRAINT pk_orders
PRIMARY KEY (order_id);

In [0]:
%sql
ALTER TABLE ecommerce.silver.orders
DROP CONSTRAINT fk_orders_customer;

In [0]:
%sql
ALTER TABLE ecommerce.silver.orders
ADD CONSTRAINT fk_orders_customer
FOREIGN KEY (customer_id)
REFERENCES ecommerce.silver.customers(customer_id)

In [0]:
%sql
DELETE FROM ecommerce.silver.orders
WHERE customer_id NOT IN (
    SELECT customer_id FROM ecommerce.silver.customers
);

In [0]:
from pyspark.sql.functions import col, trim, upper, current_timestamp, when
from datetime import datetime

start_time = datetime.now()

try:

    orders = spark.table("ecommerce.bronze.orders")

    silver_orders = (
        orders

        # Remove null foreign key
        .filter(col("customer_id").isNotNull())
        
        # Standardize order status
        .withColumn("order_status", upper(trim(col("order_status"))))
        
        # Fix carrier_date > customer_date (swap)
        .withColumn(
            "order_delivered_carrier_date_fixed",
            when(
                col("order_delivered_carrier_date") > col("order_delivered_customer_date"),
                col("order_delivered_customer_date")
            ).otherwise(col("order_delivered_carrier_date"))
        )
        
        .withColumn(
            "order_delivered_customer_date_fixed",
            when(
                col("order_delivered_carrier_date") > col("order_delivered_customer_date"),
                col("order_delivered_carrier_date")
            ).otherwise(col("order_delivered_customer_date"))
        )

        # Fix approved_at > carrier_date
        .withColumn(
            "order_approved_at_fixed",
            when(
                col("order_approved_at") > col("order_delivered_carrier_date_fixed"),
                col("order_delivered_carrier_date_fixed")
            ).otherwise(col("order_approved_at"))
        )

        # Fix delivered status but null customer delivery date
        .withColumn(
            "customer_fixed",
            when(
                (col("order_status") == "DELIVERED") &
                (col("order_delivered_customer_date_fixed").isNull()),
                col("order_delivered_carrier_date_fixed")
            ).otherwise(col("order_delivered_customer_date_fixed"))
        )
        
        # Metadata columns
        .withColumn("create_date", current_timestamp())
        .withColumn("update_date", current_timestamp())
        .withColumn("file_location", col("source_file"))
        
        # Final column selection
        .select(
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            col("order_approved_at_fixed").alias("order_approved_at"),
            col("order_delivered_carrier_date_fixed").alias("order_delivered_carrier_date"),
            col("order_delivered_customer_date_fixed").alias("order_delivered_customer_date"),
            "order_estimated_delivery_date",
            "create_date",
            "update_date",
            "file_location"
        )
    )

    record_count = silver_orders.count()

    silver_orders.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce.silver.orders")

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "orders",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "Orders table loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "orders",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###==================================================
##REVIEWS TABLE
###==================================================

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.reviews (
    review_id STRING NOT NULL, -- Primary Key
    order_id STRING NOT NULL, -- Foreign Key
    review_score INT,
    review_comment_title STRING,
    review_comment_message STRING,
    review_creation_date TIMESTAMP,
    review_answer_timestamp TIMESTAMP,
    create_date TIMESTAMP,
    update_date TIMESTAMP,
    file_location STRING
)
USING DELTA

In [0]:
%sql
ALTER TABLE ecommerce.silver.reviews
DROP CONSTRAINT pk_reviews CASCADE;

In [0]:
%sql
ALTER TABLE ecommerce.silver.reviews
ADD CONSTRAINT pk_reviews
PRIMARY KEY (review_id);

In [0]:
%sql
ALTER TABLE ecommerce.silver.reviews
DROP CONSTRAINT fk_reviews_orders;

In [0]:
%sql
ALTER TABLE ecommerce.silver.reviews
ADD CONSTRAINT fk_reviews_orders
FOREIGN KEY (order_id)
REFERENCES ecommerce.silver.orders(order_id)

In [0]:
%sql
DELETE FROM ecommerce.silver.reviews
WHERE order_id NOT IN (
    SELECT order_id FROM ecommerce.silver.orders
);

In [0]:
from pyspark.sql.functions import col, trim, lower, current_timestamp
from datetime import datetime

start_time = datetime.now()

try:

    reviews = spark.table("ecommerce.bronze.reviews_parsed")

    silver_reviews = (
        reviews
        
        # Remove null primary key
        .filter(col("review_id").isNotNull())
        
        # Remove duplicates
        .dropDuplicates(["review_id"])

        .filter(col("order_id").isNotNull())

        # Standardization
        .withColumn("review_comment_title", lower(trim(col("review_comment_title"))))
        .withColumn("review_comment_message", lower(trim(col("review_comment_message"))))
        
        # Metadata columns
        .withColumn("create_date", current_timestamp())
        .withColumn("update_date", current_timestamp())
        .withColumn("file_location", col("source_file"))
        
        # Select final columns
        .select(
            "review_id",
            "order_id",
            "review_score",
            "review_comment_title",
            "review_comment_message",
            "review_creation_date",
            "review_answer_timestamp",
            "create_date",
            "update_date",
            "file_location"
        )
    )

    record_count = silver_reviews.count()

    silver_reviews.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce.silver.reviews")

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "reviews",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "Reviews table loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "reviews",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###==================================================
##CATEGORY TABLE
###==================================================

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.category (
  product_category_name STRING NOT NULL, --Primary Key, Foreign Key
  product_category_name_english STRING,
  create_date TIMESTAMP,
  update_date TIMESTAMP,
  file_location STRING
)

In [0]:
%sql
ALTER TABLE ecommerce.silver.category
DROP CONSTRAINT pk_category CASCADE;

In [0]:
%sql
ALTER TABLE ecommerce.silver.category
ADD CONSTRAINT pk_category
PRIMARY KEY (product_category_name);

In [0]:
from pyspark.sql.functions import col,lower, current_timestamp
from datetime import datetime

start_time = datetime.now()

try:

    category = spark.table("ecommerce.bronze.category")

    silver_category = (
        category
        
        # Standardization
        .withColumn("product_category_name", lower(trim(col("product_category_name"))))
        .withColumn("product_category_name_english", lower(trim(col("product_category_name_english"))))

        # Metadata columns
        .withColumn("create_date", current_timestamp())
        .withColumn("update_date", current_timestamp())
        .withColumn("file_location", col("source_file"))
        
        # Select final columns
        .select(
            "product_category_name",
            "product_category_name_english",
            "create_date",
            "update_date",
            "file_location"
        )
    )

    record_count = silver_category.count()

    silver_category.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce.silver.category")

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "category",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "category table loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "category",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###==================================================
##PRODUCT TABLE
###==================================================

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.product (
    product_id STRING NOT NULL, -- Primary Key
    product_category_name STRING NOT NULL, -- Foreign Key
    product_name_lenght INT,
    product_description_lenght INT,
    product_photos_qty INT,
    product_weight_g INT,
    product_length_cm INT,
    product_height_cm INT,
    product_width_cm INT,
    create_date TIMESTAMP,
    update_date TIMESTAMP,
    file_location STRING
)
USING DELTA

In [0]:
%sql
ALTER TABLE ecommerce.silver.product
DROP CONSTRAINT pk_product CASCADE;

In [0]:
%sql
ALTER TABLE ecommerce.silver.product
ADD CONSTRAINT pk_product
PRIMARY KEY (product_id);

In [0]:
%sql
ALTER TABLE ecommerce.silver.product
DROP CONSTRAINT fk_product_category;

In [0]:
%sql
ALTER TABLE ecommerce.silver.product
ADD CONSTRAINT fk_product_category
FOREIGN KEY (product_category_name)
REFERENCES ecommerce.silver.category(product_category_name)

In [0]:
%sql
DELETE FROM ecommerce.silver.product
WHERE product_category_name NOT IN (
    SELECT product_category_name FROM ecommerce.silver.category
);

In [0]:
from pyspark.sql.functions import col, lower,  current_timestamp
from datetime import datetime

start_time = datetime.now()

try:

    product = spark.table("ecommerce.bronze.products")

    # Identify Bad Records
    bad_records = product.filter(col("product_category_name").isNull())

    bad_record_count = bad_records.count()

    bad_records = (
        bad_records
        .withColumn("error_reason", 
                    col("product_category_name").isNull().cast("string"))
        .withColumn("error_timestamp", current_timestamp())
        .withColumn("file_location", col("source_file"))

        # Standardization
        .withColumn("product_category_name", lower(trim(col("product_category_name"))))

    )

    bad_records.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce.silver.product_bad_records")

    # Clean Valid Records

    silver_product = (
        product
        
        # Remove bad records
        .filter(col("product_category_name").isNotNull())

        # Standardization
        .withColumn("product_category_name", lower(trim(col("product_category_name"))))
        
        # Metadata columns
        .withColumn("create_date", current_timestamp())
        .withColumn("update_date", current_timestamp())
        .withColumn("file_location", col("source_file"))
        
        # Final schema
        .select(
            "product_id",
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
            "create_date",
            "update_date",
            "file_location"
        )
    )

    record_count = silver_product.count()

    silver_product.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce.silver.product")


    end_time = datetime.now()

    log_pipeline(
        "silver",
        "product",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        f"product table loaded successfully, Bad records: {bad_record_count}"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "product",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###==================================================
##SELLERS TABLE
###==================================================

In [0]:

%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.sellers (
  seller_id STRING NOT NULL, --Primary Key, Foreign Key
  seller_zip_code_prefix INT,
  seller_city STRING,
  seller_state STRING,
  create_date TIMESTAMP,
  update_date TIMESTAMP,
  file_location STRING
)

In [0]:
%sql
ALTER TABLE ecommerce.silver.sellers
DROP CONSTRAINT pk_sellers CASCADE;

In [0]:
%sql
ALTER TABLE ecommerce.silver.sellers
ADD CONSTRAINT pk_sellers
PRIMARY KEY (seller_id);

In [0]:
from pyspark.sql.functions import col, upper,  current_timestamp
from datetime import datetime

start_time = datetime.now()

try:

    sellers = spark.table("ecommerce.bronze.sellers")

    # Clean Valid Records

    silver_sellers = (
        sellers
        
        # Standardization
        .withColumn("seller_city", upper(trim(col("seller_city"))))
        .withColumn("seller_state", upper(trim(col("seller_state"))))

        
        # Metadata columns
        .withColumn("create_date", current_timestamp())
        .withColumn("update_date", current_timestamp())
        .withColumn("file_location", col("source_file"))
        
        # Final schema
        .select(
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state",
            "create_date",
            "update_date",
            "file_location"
        )
    )

    record_count = silver_sellers.count()

    silver_sellers.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce.silver.sellers")

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "sellers",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "sellers table loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "sellers",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###==================================================
##GEOLOCATION TABLE
###==================================================

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.geolocation (
    
    geolocation_zip_code_prefix INT NOT NULL,
    geolocation_lat DOUBLE,
    geolocation_lng DOUBLE,
    geolocation_city STRING,
    geolocation_state STRING,
    create_date TIMESTAMP,
    update_date TIMESTAMP,
    file_location STRING

)
USING DELTA
COMMENT 'Cleaned geolocation data aggregated by zip_code_prefix'

In [0]:
from pyspark.sql.functions import col, upper, trim, current_timestamp, avg, first
from datetime import datetime

start_time = datetime.now()

try:
    geo = spark.table("ecommerce.bronze.geolocation")

    silver_geo = (
        geo
        # Standardization
        .withColumn("geolocation_city", upper(trim(col("geolocation_city"))))
        .withColumn("geolocation_state", upper(trim(col("geolocation_state"))))
        # Deduplicate by zip code and aggregate
        .groupBy(
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state"
        )
        .agg(
            avg("geolocation_lat").alias("geolocation_lat"),
            avg("geolocation_lng").alias("geolocation_lng"),
            first("source_file").alias("file_location")
        )
        # Metadata
        .withColumn("create_date", current_timestamp())
        .withColumn("update_date", current_timestamp())
        # Final schema
        .select(
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng",
            "geolocation_city",
            "geolocation_state",
            "create_date",
            "update_date",
            "file_location"
        )
    )

    record_count = silver_geo.count()

    silver_geo.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce.silver.geolocation")

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "geolocation",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "geolocation table loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "geolocation",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )
    raise

###==================================================
##ORDER_ITEMS TABLE
###==================================================

In [0]:
%sql
DROP TABLE ecommerce.silver.order_items;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.order_items (

    order_id STRING NOT NULL,
    order_item_id INT NOT NULL,
    product_id STRING NOT NULL,
    seller_id STRING NOT NULL,
    shipping_limit_date TIMESTAMP,
    price DOUBLE NOT NULL,
    freight_value DOUBLE NOT NULL,
    create_date TIMESTAMP,
    update_date TIMESTAMP,
    file_location STRING,
    

    CONSTRAINT pk_order_items 
        PRIMARY KEY (order_id, order_item_id),

    CONSTRAINT fk_order_items_orders
        FOREIGN KEY (order_id)
        REFERENCES ecommerce.silver.orders(order_id),

    CONSTRAINT fk_order_items_products
        FOREIGN KEY (product_id)
        REFERENCES ecommerce.silver.product(product_id),

    CONSTRAINT fk_order_items_sellers
        FOREIGN KEY (seller_id)
        REFERENCES ecommerce.silver.sellers(seller_id)

)
USING DELTA
COMMENT 'Order line items table containing product-level details for each order'

In [0]:
from pyspark.sql.functions import col,  current_timestamp
from datetime import datetime

start_time = datetime.now()

try:
    # Load bronze tables
    order_items = spark.table("ecommerce.bronze.order_items")

    # Remove null primary key
    order_items_clean = order_items.filter(
        col("order_id").isNotNull() &
        col("order_item_id").isNotNull()
    )

    # Remove duplicate records
    order_items_clean = order_items_clean.dropDuplicates(
        ["order_id", "order_item_id"]
    )

    # Clean Valid Records
    silver_order_items = (
        order_items_clean
        
        # Metadata columns
        .withColumn("create_date", current_timestamp())
        .withColumn("update_date", current_timestamp())
        .withColumn("file_location", col("source_file"))
        
        # Final schema
        .select(
            "order_id",
            "order_item_id",
            "product_id",
            "seller_id",
            "shipping_limit_date",
            "price",
            "freight_value",
            "create_date",
            "update_date",
            "file_location"
        )
    )
    record_count = silver_order_items.count()

    silver_order_items.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce.silver.order_items")

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "order_items",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "order_items table loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "order_items",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###==================================================
##PAYMENTS TABLE
###==================================================

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.payments(
  order_id STRING NOT NULL,
  payment_sequential INT NOT NULL,
  payment_type STRING,
  payment_installments INT,
  payment_value DOUBLE,
  create_date TIMESTAMP,
  update_date TIMESTAMP,
  file_location STRING
)

USING DELTA

In [0]:
%sql
ALTER TABLE ecommerce.silver.payments
ADD CONSTRAINT pk_payments
PRIMARY KEY (order_id, payment_sequential)

In [0]:
%sql
ALTER TABLE ecommerce.silver.payments
ADD CONSTRAINT fk_payments
FOREIGN KEY (order_id)
REFERENCES ecommerce.silver.orders(order_id)

In [0]:
%sql
DELETE FROM ecommerce.silver.payments
WHERE order_id NOT IN (
    SELECT order_id FROM ecommerce.silver.orders
);

In [0]:
spark.sql("USE CATALOG ecommerce")
spark.sql("USE SCHEMA bronze")
spark.sql("USE SCHEMA silver")

In [0]:
bron_payments = spark.table("ecommerce.bronze.payments")
bron_payments.filter("order_id IS NULL").count()

In [0]:
bron_payments.filter("payment_sequential IS NULL").count()

In [0]:
from pyspark.sql.functions import col, upper,  current_timestamp,trim
from datetime import datetime

start_time = datetime.now()

try:

    payments = spark.table("ecommerce.bronze.payments")
    orders = spark.table("ecommerce.silver.orders")

    # Remove null PK
    payments = payments.filter(
        col("order_id").isNotNull() &
        col("payment_sequential").isNotNull()
    )

    # Remove duplicates
    payments = payments.dropDuplicates(
        ["order_id", "payment_sequential"]
    )

    # Validate numeric values
    payments = payments.filter(
        (col("payment_value") >= 0) &
        (col("payment_installments") >= 0)
    )

    # FK validation
    valid_records = payments.join(
        orders.select("order_id"),
        "order_id",
        "inner"
    )

    silver_payments = (
        valid_records
        
        # Standardization
        .withColumn("payment_type", upper(trim(col("payment_type"))))
        
        # Metadata columns
        .withColumn("create_date", current_timestamp())
        .withColumn("update_date", current_timestamp())
        .withColumn("file_location", col("source_file"))
        
        # Final schema
        .select(
            "order_id",
            "payment_sequential",
            "payment_type",
            "payment_installments",
            "payment_value",
            "create_date",
            "update_date",
            "file_location"
        )
    )

    record_count = silver_payments.count()

    silver_payments.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("ecommerce.silver.payments")

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "payments",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "payments table loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "silver",
        "payments",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise